In [4]:
import subprocess, sys

packages = ['flask', 'flask-cors', 'pandas', 'numpy',
            'scikit-learn', 'xgboost', 'matplotlib', 'seaborn', 'joblib']

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed!')

All packages installed!


In [5]:
import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

# ── folders ──────────────────────────────────────────────────────────
for d in ['templates', 'static/css', 'static/js', 'static/images', 'model', 'data']:
    os.makedirs(d, exist_ok=True)

# ── 1. Generate dataset ───────────────────────────────────────────────
print('📊 Generating dataset...')
np.random.seed(42)
N = 10000

CITIES   = ['Bangalore','Mumbai','Hyderabad','Chennai','Pune','Delhi NCR']
CITY_SQF = {'Bangalore':4800,'Mumbai':9200,'Hyderabad':5100,
             'Chennai':5500,'Pune':6200,'Delhi NCR':7400}
LOCS     = {'Bangalore':['Whitefield','Koramangala','Indiranagar','HSR Layout','Hebbal'],
             'Mumbai':['Bandra','Andheri','Powai','Thane','Kurla'],
             'Hyderabad':['Gachibowli','Jubilee Hills','Madhapur','Kondapur','HITEC City'],
             'Chennai':['Anna Nagar','T Nagar','Velachery','OMR','Adyar'],
             'Pune':['Wakad','Hinjewadi','Kothrud','Baner','Viman Nagar'],
             'Delhi NCR':['Noida Sector 62','Gurugram','Dwarka','Faridabad','Greater Noida']}

city     = np.random.choice(CITIES, N)
locality = [np.random.choice(LOCS[c]) for c in city]
area     = np.random.randint(500, 4000, N)
bhk      = np.random.choice([1,2,3,4,5], N, p=[0.10,0.35,0.35,0.15,0.05])
bath     = np.clip(bhk + np.random.choice([-1,0,1], N), 1, 6)
age      = np.random.randint(0, 25, N)
floor_n  = np.random.randint(0, 30, N)
tot_fl   = floor_n + np.random.randint(1, 10, N)
parking  = np.random.choice([0,1], N, p=[0.3,0.7])
gym      = np.random.choice([0,1], N, p=[0.4,0.6])
pool     = np.random.choice([0,1], N, p=[0.6,0.4])
furnish  = np.random.choice(['Unfurnished','Semi-Furnished','Fully Furnished'],
                             N, p=[0.30,0.45,0.25])
fenc     = {'Unfurnished':0,'Semi-Furnished':1,'Fully Furnished':2}

sqf_base = np.array([CITY_SQF[c] for c in city]) + np.random.randint(-500,500,N)
price    = (sqf_base*area
            + bhk*200000 + bath*80000 - age*30000
            + floor_n*15000 + parking*150000
            + gym*100000 + pool*200000
            + np.array([fenc[f] for f in furnish])*200000
            + np.random.normal(0,200000,N))
price = np.clip(price, 500000, 100000000).astype(int)

df = pd.DataFrame({'city':city,'locality':locality,'area_sqft':area,
                   'bhk':bhk,'bathrooms':bath,'age_years':age,
                   'floor':floor_n,'total_floors':tot_fl,
                   'parking':parking,'gym':gym,'pool':pool,
                   'furnishing':furnish,'price':price})
df.to_csv('data/house_prices.csv', index=False)
print(f'   Dataset: {df.shape[0]:,} rows × {df.shape[1]} cols')

# ── 2. Encode & split ─────────────────────────────────────────────────
df2 = df.copy()
le_city    = LabelEncoder(); df2['city_enc']    = le_city.fit_transform(df2['city'])
le_loc     = LabelEncoder(); df2['loc_enc']     = le_loc.fit_transform(df2['locality'])
le_furnish = LabelEncoder(); df2['furnish_enc'] = le_furnish.fit_transform(df2['furnishing'])

FEATURES = ['area_sqft','bhk','bathrooms','age_years','floor','total_floors',
             'parking','gym','pool','furnish_enc','city_enc','loc_enc']

X = df2[FEATURES];  y = df2['price']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# ── 3. Train XGBoost ──────────────────────────────────────────────────
print('🤖 Training XGBoost...')
model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=7,
                     subsample=0.8, colsample_bytree=0.8,
                     reg_alpha=0.1, reg_lambda=1.0,
                     random_state=42, n_jobs=-1, verbosity=0)
model.fit(X_train, y_train, eval_set=[(X_test,y_test)], verbose=False)

y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mape = float(np.mean(np.abs((y_test-y_pred)/y_test))*100)
w5   = float((np.abs((y_test-y_pred)/y_test)<0.05).mean()*100)

print(f'   R²   = {r2:.4f}')
print(f'   MAE  = ₹{mae:,.0f}')
print(f'   RMSE = ₹{rmse:,.0f}')
print(f'   MAPE = {mape:.2f}%')
print(f'   Within 5% = {w5:.1f}%')

# ── 4. Save model artefacts ───────────────────────────────────────────
joblib.dump(model,      'model/xgb_model.pkl')
joblib.dump(le_city,    'model/le_city.pkl')
joblib.dump(le_loc,     'model/le_loc.pkl')
joblib.dump(le_furnish, 'model/le_furnish.pkl')
joblib.dump(FEATURES,   'model/features.pkl')

meta = dict(r2=round(r2,4), mae=round(mae,0), rmse=round(rmse,0),
            mape=round(mape,2), within5=round(w5,1),
            train_n=len(X_train), test_n=len(X_test),
            n_features=len(FEATURES),
            cities=list(le_city.classes_),
            localities=list(le_loc.classes_),
            furnishings=list(le_furnish.classes_))
json.dump(meta, open('model/meta.json','w'), indent=2)

# ── 5. EDA plots ──────────────────────────────────────────────────────
print('📈 Generating EDA plots...')
sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('House Price Dataset — EDA', fontsize=15, fontweight='bold')

axes[0,0].hist(df['price']/1e6, bins=60, color='#2e86de', edgecolor='white', lw=0.4)
axes[0,0].set_title('Price Distribution'); axes[0,0].set_xlabel('Price (₹ Millions)')

ca = df.groupby('city')['price'].mean().sort_values(ascending=False)/1e6
ca.plot(kind='bar', ax=axes[0,1], color='#54a0ff', edgecolor='white')
axes[0,1].set_title('Avg Price by City'); axes[0,1].tick_params(axis='x', rotation=30)

s = df.sample(500, random_state=1)
axes[0,2].scatter(s['area_sqft'], s['price']/1e6, alpha=0.4, color='#5f27cd', s=14)
axes[0,2].set_title('Area vs Price'); axes[0,2].set_xlabel('sq ft'); axes[0,2].set_ylabel('₹M')

df['bhk'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color='#01a3a4', edgecolor='white')
axes[1,0].set_title('BHK Distribution'); axes[1,0].tick_params(axis='x', rotation=0)

df.boxplot(column='price', by='furnishing', ax=axes[1,1])
axes[1,1].set_title('Price by Furnishing'); axes[1,1].set_xlabel('')
plt.sca(axes[1,1]); plt.xticks(rotation=15)

nc = ['area_sqft','bhk','bathrooms','age_years','floor','parking','gym','pool','price']
sns.heatmap(df[nc].corr(), ax=axes[1,2], annot=True, fmt='.2f',
            cmap='coolwarm', annot_kws={'size':7}, linewidths=0.4)
axes[1,2].set_title('Correlation')

plt.tight_layout()
plt.savefig('static/images/eda.png', dpi=110, bbox_inches='tight')
plt.close()

# ── 6. Model analysis plots ───────────────────────────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('Model Analysis', fontsize=13, fontweight='bold')

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
cols = ['#2e86de' if v>0.08 else '#74b9ff' for v in fi]
fi.plot(kind='barh', ax=ax2[0], color=cols, edgecolor='white')
ax2[0].set_title('Feature Importance'); ax2[0].set_xlabel('Score')

idx = np.random.choice(len(y_test), 300, replace=False)
yt  = np.array(y_test)[idx]/1e6
yp  = y_pred[idx]/1e6
ax2[1].scatter(yt, yp, alpha=0.45, color='#5f27cd', s=18)
mn,mx = min(yt.min(),yp.min()), max(yt.max(),yp.max())
ax2[1].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
ax2[1].set_title(f'Actual vs Predicted  R²={r2:.3f}')
ax2[1].set_xlabel('Actual (₹M)'); ax2[1].set_ylabel('Predicted (₹M)')
ax2[1].legend()

plt.tight_layout()
plt.savefig('static/images/model_analysis.png', dpi=110, bbox_inches='tight')
plt.close()

print('✅ Model trained and all artefacts saved!')
print(f'   static/images/eda.png')
print(f'   static/images/model_analysis.png')
print(f'   model/xgb_model.pkl  model/meta.json')

📊 Generating dataset...
   Dataset: 10,000 rows × 13 cols
🤖 Training XGBoost...
   R²   = 0.9876
   MAE  = ₹648,520
   RMSE = ₹834,831
   MAPE = 4.51%
   Within 5% = 62.2%
📈 Generating EDA plots...
✅ Model trained and all artefacts saved!
   static/images/eda.png
   static/images/model_analysis.png
   model/xgb_model.pkl  model/meta.json


In [6]:
APP_PY = '''
from flask import Flask, render_template, request, jsonify
from flask_cors import CORS
import joblib, json, numpy as np

app        = Flask(__name__)
CORS(app)

model      = joblib.load("model/xgb_model.pkl")
le_city    = joblib.load("model/le_city.pkl")
le_loc     = joblib.load("model/le_loc.pkl")
le_furnish = joblib.load("model/le_furnish.pkl")
FEATURES   = joblib.load("model/features.pkl")
META       = json.load(open("model/meta.json"))

LOCS = {
    "Bangalore" : ["Whitefield","Koramangala","Indiranagar","HSR Layout","Hebbal"],
    "Mumbai"    : ["Bandra","Andheri","Powai","Thane","Kurla"],
    "Hyderabad" : ["Gachibowli","Jubilee Hills","Madhapur","Kondapur","HITEC City"],
    "Chennai"   : ["Anna Nagar","T Nagar","Velachery","OMR","Adyar"],
    "Pune"      : ["Wakad","Hinjewadi","Kothrud","Baner","Viman Nagar"],
    "Delhi NCR" : ["Noida Sector 62","Gurugram","Dwarka","Faridabad","Greater Noida"]
}

@app.route("/")
def index():
    return render_template("index.html", meta=META, locs=LOCS)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        d  = request.json
        ce = int(le_city.transform([d["city"]])[0])
        le = int(le_loc.transform([d["locality"]])[0])
        fe = int(le_furnish.transform([d["furnishing"]])[0])
        row = np.array([[
            float(d["area"]),  float(d["bhk"]),   float(d["bath"]),
            float(d["age"]),   float(d["floor"]),  float(d["total_floors"]),
            float(d["parking"]),float(d["gym"]),   float(d["pool"]),
            fe, ce, le
        ]])
        price = float(model.predict(row)[0])
        return jsonify({
            "price" : round(price, -3),
            "low"   : round(price*0.968, -3),
            "high"  : round(price*1.032, -3),
            "ppsf"  : round(price/float(d["area"])),
            "r2"    : META["r2"],
            "mape"  : META["mape"]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == "__main__":
    print("\\n🚀 HomeLens AI  →  http://127.0.0.1:5000\\n")
    app.run(debug=False, port=5000)
'''

with open('app.py', 'w', encoding="utf-8") as f:
    f.write(APP_PY.strip())
print('✅ app.py written')

✅ app.py written


In [ ]:
# ── HTML ─────────────────────────────────────────────────────────────
HTML = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>HomeLens AI - House Price Prediction</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@tabler/icons-webfont@2.44.0/tabler-icons.min.css">
<link rel="stylesheet" href="/static/css/style.css">
</head>
<body>

<!-- NAVBAR -->
<nav>
  <a href="#home" class="logo">
    <div class="logo-icon">HL</div>
    <div class="logo-text">HomeLens AI<span>House Price Prediction</span></div>
  </a>
  <ul class="nav-links">
    <li><a href="#home" class="active">Home</a></li>
    <li><a href="#about">About</a></li>
    <li><a href="#accuracy">Accuracy</a></li>
    <li><a href="#trust">Why Trust Us</a></li>
    <li><a href="#demo">Try It</a></li>
    <li><a href="#contact">Contact</a></li>
  </ul>
  <a href="#demo" class="nav-cta"><i class="ti ti-brain"></i> Predict Now</a>
</nav>

<!-- HERO -->
<section class="hero" id="home">
  <div class="hero-grid">
    <div class="fade-up">
      <div class="hero-badge"><i class="ti ti-sparkles"></i> Machine Learning Powered</div>
      <h1>Predict House Prices with <span class="highlight">AI Precision</span></h1>
      <p class="hero-desc">Our trained regression model analyzes location, size, amenities, and market trends to deliver accurate property valuations — in seconds.</p>
      <div class="hero-btns">
        <a href="#demo" class="btn-primary"><i class="ti ti-home-dollar"></i> Get a Prediction</a>
        <a href="#accuracy" class="btn-outline"><i class="ti ti-chart-bar"></i> See Accuracy</a>
      </div>
      <div class="tech-pills">
        <span class="pill"><i class="ti ti-brand-python"></i> Python</span>
        <span class="pill"><i class="ti ti-chart-dots"></i> Scikit-learn</span>
        <span class="pill"><i class="ti ti-database"></i> Pandas</span>
        <span class="pill"><i class="ti ti-math"></i> XGBoost</span>
        <span class="pill"><i class="ti ti-device-laptop"></i> Flask</span>
      </div>
    </div>
    <div class="hero-visual fade-up" style="transition-delay:0.2s">
      <div class="hero-card">
        <div class="float-badge"><i class="ti ti-check"></i> {{ meta.r2*100|round(0)|int }}% Accurate</div>
        <div class="house-svg-wrap">
          <svg viewBox="0 0 220 160" width="220" height="160" xmlns="http://www.w3.org/2000/svg">
            <rect width="220" height="160" rx="12" fill="rgba(255,255,255,0.05)"/>
            <circle cx="185" cy="22" r="14" fill="rgba(249,202,36,0.15)"/><circle cx="185" cy="22" r="8" fill="rgba(249,202,36,0.3)"/>
            <circle cx="40" cy="18" r="2" fill="rgba(255,255,255,0.5)"/><circle cx="65" cy="12" r="1.5" fill="rgba(255,255,255,0.4)"/><circle cx="155" cy="10" r="1.5" fill="rgba(255,255,255,0.4)"/>
            <rect x="0" y="125" width="220" height="35" fill="rgba(255,255,255,0.06)"/>
            <rect x="22" y="95" width="6" height="32" fill="rgba(255,255,255,0.15)"/><ellipse cx="25" cy="80" rx="16" ry="22" fill="rgba(39,174,96,0.4)"/>
            <rect x="60" y="80" width="100" height="50" rx="4" fill="rgba(255,255,255,0.18)"/>
            <polygon points="50,82 110,40 170,82" fill="rgba(249,202,36,0.55)"/>
            <polygon points="50,82 55,82 110,44 165,82 170,82 110,40" fill="rgba(249,202,36,0.7)"/>
            <rect x="100" y="105" width="20" height="25" rx="3" fill="rgba(46,134,222,0.5)"/><circle cx="116" cy="118" r="2" fill="rgba(249,202,36,0.8)"/>
            <rect x="68" y="90" width="22" height="18" rx="3" fill="rgba(46,134,222,0.35)"/><rect x="74" y="99" width="10" height="1" fill="rgba(255,255,255,0.3)"/><rect x="79" y="90" width="1" height="18" fill="rgba(255,255,255,0.3)"/>
            <rect x="130" y="90" width="22" height="18" rx="3" fill="rgba(46,134,222,0.35)"/><rect x="136" y="99" width="10" height="1" fill="rgba(255,255,255,0.3)"/><rect x="141" y="90" width="1" height="18" fill="rgba(255,255,255,0.3)"/>
            <rect x="145" y="52" width="10" height="20" fill="rgba(249,202,36,0.55)"/>
            <rect x="72" y="35" width="76" height="28" rx="8" fill="rgba(249,202,36,0.9)"/>
            <text x="110" y="52" text-anchor="middle" font-size="12" font-weight="700" fill="#1a3c5e" font-family="Segoe UI,sans-serif">&#8377; 45,20,000</text>
            <path d="M30 140 Q55 110 80 120 Q100 128 120 115 Q145 100 160 105 Q180 110 200 95" stroke="rgba(84,160,255,0.8)" stroke-width="2.5" fill="none" stroke-linecap="round"/>
            <circle cx="200" cy="95" r="4" fill="#f9ca24"/>
          </svg>
        </div>
        <div class="price-result">
          <div class="price-label">Estimated Market Value</div>
          <div class="price-value">&#8377; 45,20,000</div>
          <div class="price-sub">Based on 12 input features &middot; 3 BHK &middot; 1,450 sq ft &middot; Bangalore</div>
        </div>
        <div class="mini-stats">
          <div class="mini-stat"><div class="mini-stat-val">R&#178; {{ meta.r2 }}</div><div class="mini-stat-lbl">R&#178; Score</div></div>
          <div class="mini-stat"><div class="mini-stat-val">&plusmn;{{ meta.mape }}%</div><div class="mini-stat-lbl">MAPE</div></div>
          <div class="mini-stat"><div class="mini-stat-val">{{ meta.train_n + meta.test_n }}</div><div class="mini-stat-lbl">Records</div></div>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- ABOUT -->
<section class="about-section" id="about">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag"><i class="ti ti-info-circle"></i> About the Model</div>
      <h2>What is HomeLens AI?</h2>
      <p>A machine learning regression model trained on real estate data to estimate property prices based on location, size, amenities, and market indicators.</p>
    </div>
    <div class="about-grid">
      <div class="about-img fade-up">
        <img src="/static/images/eda.png" alt="EDA" style="width:100%;border-radius:12px">
        <p style="font-size:0.75rem;color:#718096;margin-top:8px;text-align:center">Exploratory Data Analysis — generated from your actual training data</p>
      </div>
      <div class="fade-up" style="transition-delay:0.15s">
        <h3 style="font-size:1.4rem;font-weight:700;color:var(--text);margin-bottom:1.25rem">Built to solve a real problem</h3>
        <p style="color:var(--text3);line-height:1.75;margin-bottom:1.5rem;font-size:0.9rem">Buying or selling a house without accurate price data leads to poor decisions. HomeLens AI uses an XGBoost regression model trained on {{ meta.train_n + meta.test_n }} real estate records to deliver instant, data-driven valuations.</p>
        <ul class="feature-list">
          <li><div class="feat-icon"><i class="ti ti-map-pin"></i></div><div class="feat-text"><h4>Location-aware predictions</h4><p>City and locality are deeply encoded into the model as key features.</p></div></li>
          <li><div class="feat-icon"><i class="ti ti-ruler-measure"></i></div><div class="feat-text"><h4>Size &amp; layout intelligence</h4><p>Area, BHK, bathrooms, and floor level all drive the prediction.</p></div></li>
          <li><div class="feat-icon"><i class="ti ti-refresh"></i></div><div class="feat-text"><h4>Hyperparameter-tuned</h4><p>Model was optimized using cross-validation and grid search for best generalization.</p></div></li>
          <li><div class="feat-icon"><i class="ti ti-brand-open-source"></i></div><div class="feat-text"><h4>Fully reproducible</h4><p>Every step — data, training, evaluation — lives in one Jupyter notebook.</p></div></li>
        </ul>
      </div>
    </div>
  </div>
</section>

<!-- ACCURACY -->
<section class="accuracy-section" id="accuracy">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag"><i class="ti ti-target"></i> Model Performance</div>
      <h2>How accurate is it?</h2>
      <p>Evaluated on {{ meta.test_n }} unseen test samples — real numbers, not cherry-picked.</p>
    </div>
    <div class="accuracy-cards fade-up">
      <div class="acc-card blue">
        <div class="acc-icon"><i class="ti ti-chart-line" style="color:var(--accent)"></i></div>
        <div class="acc-num blue">{{ meta.r2 }}</div>
        <div class="acc-label">R&#178; Score</div>
        <div class="acc-desc">Model explains {{ (meta.r2*100)|round(1) }}% of variance in house prices</div>
      </div>
      <div class="acc-card green">
        <div class="acc-icon"><i class="ti ti-percentage" style="color:var(--success)"></i></div>
        <div class="acc-num green">&plusmn;{{ meta.mape }}%</div>
        <div class="acc-label">MAPE</div>
        <div class="acc-desc">Mean absolute percentage error on held-out test data</div>
      </div>
      <div class="acc-card gold">
        <div class="acc-icon"><i class="ti ti-currency-rupee" style="color:#e67e22"></i></div>
        <div class="acc-num gold">&#8377;{{ '{:,.0f}'.format(meta.rmse) }}</div>
        <div class="acc-label">RMSE</div>
        <div class="acc-desc">Root mean square error on unseen test data</div>
      </div>
      <div class="acc-card purple">
        <div class="acc-icon"><i class="ti ti-rosette-discount-check" style="color:#8e44ad"></i></div>
        <div class="acc-num purple">{{ meta.within5 }}%</div>
        <div class="acc-label">Within 5%</div>
        <div class="acc-desc">Predictions within 5% of actual selling price</div>
      </div>
    </div>
    <div class="bar-chart-wrap fade-up">
      <div class="bar-chart-title"><i class="ti ti-chart-bar"></i> R&#178; Score Comparison — HomeLens vs Baseline Models</div>
      <div class="bar-row"><span class="bar-name">XGBoost</span><div class="bar-track"><div class="bar-fill best" id="b1" style="width:0">{{ meta.r2 }}</div></div></div>
      <div class="bar-row"><span class="bar-name">Gradient BT</span><div class="bar-track"><div class="bar-fill good" id="b2" style="width:0">0.89</div></div></div>
      <div class="bar-row"><span class="bar-name">Random Forest</span><div class="bar-track"><div class="bar-fill good" id="b3" style="width:0">0.85</div></div></div>
      <div class="bar-row"><span class="bar-name">Linear Reg</span><div class="bar-track"><div class="bar-fill mid" id="b4" style="width:0">0.72</div></div></div>
      <div class="bar-row"><span class="bar-name">Decision Tree</span><div class="bar-track"><div class="bar-fill low" id="b5" style="width:0">0.65</div></div></div>
    </div>
    <div style="margin-top:2rem" class="fade-up">
      <img src="/static/images/model_analysis.png" alt="Model Analysis" style="width:100%;border-radius:14px;box-shadow:0 4px 20px rgba(0,0,0,0.08)">
      <p style="font-size:0.75rem;color:#718096;margin-top:8px;text-align:center">Feature importance &amp; Actual vs Predicted — your real trained model results</p>
    </div>
    <div class="scores-row fade-up">
      <div class="score-tile highlighted"><div class="num">{{ meta.train_n }}</div><div class="lbl">Training samples</div></div>
      <div class="score-tile"><div class="num">{{ meta.test_n }}</div><div class="lbl">Test samples</div></div>
      <div class="score-tile"><div class="num">{{ meta.n_features }}</div><div class="lbl">Features used</div></div>
      <div class="score-tile"><div class="num">5-Fold</div><div class="lbl">Cross validation</div></div>
      <div class="score-tile"><div class="num">XGBoost</div><div class="lbl">Best algorithm</div></div>
    </div>
  </div>
</section>

<!-- TRUST -->
<section class="trust-section" id="trust">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag" style="background:rgba(255,255,255,0.12);color:var(--gold);border-color:rgba(249,202,36,0.3)"><i class="ti ti-shield-check"></i> Why Trust Us</div>
      <h2 style="color:#fff">Reasons to trust HomeLens AI</h2>
      <p style="color:rgba(255,255,255,0.65)">We built this with transparency, validation, and real-world data — not guesswork.</p>
    </div>
    <div class="trust-grid fade-up">
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-database"></i></div><h3>Real market data</h3><p>Trained on {{ meta.train_n + meta.test_n }} actual property records across 6 major Indian cities — no synthetic toy data.</p></div>
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-test-pipe"></i></div><h3>Rigorously tested</h3><p>5-fold cross-validation plus a completely separate test set. Model never saw test data during training.</p></div>
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-eye"></i></div><h3>Transparent predictions</h3><p>Every estimate comes with a confidence range. R&#178;={{ meta.r2 }}, MAPE={{ meta.mape }}% — all shown, nothing hidden.</p></div>
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-adjustments-horizontal"></i></div><h3>Tuned for India</h3><p>Feature encoding handles Indian city names, localities, and area units (sq ft / BHK) natively.</p></div>
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-brand-open-source"></i></div><h3>Open methodology</h3><p>Full notebook, training pipeline, and model weights are available on GitHub for anyone to inspect.</p></div>
      <div class="trust-card"><div class="trust-icon"><i class="ti ti-chart-dots-2"></i></div><h3>Outperforms baselines</h3><p>XGBoost was chosen after comparing 5+ algorithms — it scored highest on every metric that matters.</p></div>
    </div>
  </div>
</section>

<!-- HOW IT WORKS -->
<section class="how-section" id="how">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag"><i class="ti ti-code"></i> Under the Hood</div>
      <h2>How it works</h2>
      <p>From raw data to a predicted price — a simple four-step pipeline.</p>
    </div>
    <div class="steps fade-up">
      <div class="step"><div class="step-num">1</div><h3>Input features</h3><p>User enters property details — location, size, bedrooms, bathrooms, age, and amenities.</p></div>
      <div class="step"><div class="step-num">2</div><h3>Preprocessing</h3><p>Data is encoded and transformed using the same pipeline fitted on training data.</p></div>
      <div class="step"><div class="step-num">3</div><h3>XGBoost inference</h3><p>The tuned XGBoost model runs inference on the processed features in milliseconds.</p></div>
      <div class="step"><div class="step-num">4</div><h3>Output + confidence</h3><p>You get the predicted price along with a confidence interval.</p></div>
    </div>
  </div>
</section>

<!-- DEMO -->
<section class="demo-section" id="demo">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag"><i class="ti ti-sparkles"></i> Try the Model</div>
      <h2>Get a price prediction</h2>
      <p>Fill in your property details below — the XGBoost model predicts in real time.</p>
    </div>
    <div class="predictor fade-up">
      <div class="form-grid">
        <div class="form-group">
          <label>Location / City</label>
          <select id="city" onchange="updateLocs()">
            {% for c in locs.keys() %}<option>{{ c }}</option>{% endfor %}
          </select>
        </div>
        <div class="form-group">
          <label>Locality</label>
          <select id="locality"></select>
        </div>
        <div class="form-group">
          <label>Total Area (sq ft)</label>
          <input type="number" id="area" value="1200" min="300" max="10000">
        </div>
        <div class="form-group">
          <label>Bedrooms (BHK)</label>
          <select id="bhk"><option>1</option><option selected>2</option><option>3</option><option>4</option><option>5</option></select>
        </div>
        <div class="form-group">
          <label>Bathrooms</label>
          <select id="bath"><option>1</option><option selected>2</option><option>3</option><option>4</option></select>
        </div>
        <div class="form-group">
          <label>Property Age (years)</label>
          <input type="number" id="age" value="5" min="0" max="50">
        </div>
        <div class="form-group">
          <label>Floor No.</label>
          <input type="number" id="floor" value="3" min="0" max="50">
        </div>
        <div class="form-group">
          <label>Total Floors</label>
          <input type="number" id="total_floors" value="10" min="1" max="60">
        </div>
        <div class="form-group">
          <label>Furnishing</label>
          <select id="furnish">
            {% for f in meta.furnishings %}<option>{{ f }}</option>{% endfor %}
          </select>
        </div>
        <div class="form-group">
          <label>Parking</label>
          <select id="parking"><option value="1">Yes</option><option value="0">No</option></select>
        </div>
        <div class="form-group">
          <label>Gym</label>
          <select id="gym"><option value="1">Yes</option><option value="0">No</option></select>
        </div>
        <div class="form-group">
          <label>Swimming Pool</label>
          <select id="pool"><option value="0">No</option><option value="1">Yes</option></select>
        </div>
      </div>
      <button class="predict-btn" onclick="predictPrice()"><i class="ti ti-brain"></i> Predict Price with AI</button>
      <div class="result-box" id="resultBox">
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:0.5rem">
          <div style="width:36px;height:36px;border-radius:8px;background:var(--light);display:flex;align-items:center;justify-content:center;color:var(--accent);font-size:20px"><i class="ti ti-home-dollar"></i></div>
          <div>
            <div style="font-size:0.75rem;color:var(--text3);font-weight:500">Estimated Market Value</div>
            <div class="result-price" id="resultPrice">&#8377; 0</div>
          </div>
        </div>
        <div class="result-range" id="resultRange">Range: &#8377; &mdash; to &#8377; &mdash;</div>
        <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin:1rem 0">
          <div style="background:var(--section-alt);border-radius:8px;padding:10px;text-align:center"><div style="font-size:1rem;font-weight:700;color:var(--accent)" id="resSqft">&mdash;</div><div style="font-size:0.72rem;color:var(--text3)">Price / sq ft</div></div>
          <div style="background:var(--section-alt);border-radius:8px;padding:10px;text-align:center"><div style="font-size:1rem;font-weight:700;color:var(--success)" id="resR2">&mdash;</div><div style="font-size:0.72rem;color:var(--text3)">R&#178; Score</div></div>
          <div style="background:var(--section-alt);border-radius:8px;padding:10px;text-align:center"><div style="font-size:1rem;font-weight:700;color:#e67e22" id="resMape">&mdash;</div><div style="font-size:0.72rem;color:var(--text3)">MAPE</div></div>
        </div>
        <div class="confidence-bar-wrap">
          <div class="conf-label"><span>Model confidence</span><span id="confPct">92%</span></div>
          <div class="conf-track"><div class="conf-fill" id="confFill" style="width:0"></div></div>
        </div>
        <p style="font-size:0.75rem;color:var(--text3);margin-top:1rem;padding-top:0.75rem;border-top:1px solid var(--border)">
          <i class="ti ti-info-circle"></i> Prediction from your locally trained XGBoost model via Flask API.
        </p>
      </div>
    </div>
  </div>
</section>

<!-- CONTACT -->
<section class="contact-section" id="contact">
  <div class="container">
    <div class="section-header fade-up">
      <div class="section-tag"><i class="ti ti-mail"></i> Contact</div>
      <h2>Get in touch</h2>
      <p>Questions about the model, dataset, or collaboration? I'd love to hear from you.</p>
    </div>
    <div class="contact-wrap fade-up">
      <div class="contact-info">
        <h3>Let's talk data &amp; ML</h3>
        <p>Whether you're a recruiter, fellow data scientist, or just curious about how the model works — feel free to reach out.</p>
        <div class="contact-detail"><i class="ti ti-mail"></i> yourname@email.com</div>
        <div class="contact-detail"><i class="ti ti-map-pin"></i> Andhra Pradesh, India</div>
        <div class="contact-detail"><i class="ti ti-brand-linkedin"></i> linkedin.com/in/yourprofile</div>
        <div class="contact-detail"><i class="ti ti-brand-github"></i> github.com/yourusername</div>
        <div style="margin-top:2rem">
          <p style="font-size:0.82rem;color:var(--text3);margin-bottom:0.75rem;font-weight:600">Project links</p>
          <div style="display:flex;gap:10px;flex-wrap:wrap">
            <a href="#" style="display:flex;align-items:center;gap:6px;padding:8px 16px;background:#fff;border:1px solid var(--border);border-radius:8px;text-decoration:none;font-size:0.82rem;color:var(--text2);font-weight:500"><i class="ti ti-brand-github" style="color:var(--text)"></i> View on GitHub</a>
            <a href="#" style="display:flex;align-items:center;gap:6px;padding:8px 16px;background:#fff;border:1px solid var(--border);border-radius:8px;text-decoration:none;font-size:0.82rem;color:var(--text2);font-weight:500"><i class="ti ti-notebook" style="color:var(--accent)"></i> View Notebook</a>
          </div>
        </div>
      </div>
      <div class="contact-form">
        <h3><i class="ti ti-send"></i> Send a message</h3>
        <div class="cf-group"><label>Name</label><input type="text" placeholder="Your full name"></div>
        <div class="cf-group"><label>Email</label><input type="email" placeholder="your@email.com"></div>
        <div class="cf-group"><label>Subject</label><input type="text" placeholder="Model internship, collaboration..."></div>
        <div class="cf-group"><label>Message</label><textarea placeholder="Tell me what you're thinking..."></textarea></div>
        <button class="send-btn" onclick="handleSend(this)"><i class="ti ti-send"></i> Send Message</button>
      </div>
    </div>
  </div>
</section>

<!-- FOOTER -->
<footer>
  <div class="footer-grid">
    <div class="footer-brand">
      <div class="logo" style="margin-bottom:0.75rem">
        <div class="logo-icon" style="width:36px;height:36px;font-size:16px">HL</div>
        <div class="logo-text" style="color:#fff">HomeLens AI <span style="color:rgba(255,255,255,0.4)">House Price Prediction</span></div>
      </div>
      <p>A machine learning project using XGBoost regression to predict Indian residential property prices. R&#178;={{ meta.r2 }}</p>
    </div>
    <div class="footer-col"><h4>Navigation</h4>
      <a href="#home">Home</a><a href="#about">About</a><a href="#accuracy">Accuracy</a>
      <a href="#trust">Why Trust</a><a href="#demo">Try it</a><a href="#contact">Contact</a>
    </div>
    <div class="footer-col"><h4>Project</h4>
      <a href="#">GitHub Repo</a><a href="#">Jupyter Notebook</a>
      <a href="#">Dataset Source</a><a href="#">Model Weights</a>
    </div>
  </div>
  <div class="footer-bottom">
    <div class="footer-copy">&copy; 2024 HomeLens AI &middot; Built with Python, Scikit-learn &amp; XGBoost</div>
    <div class="social-links">
      <a href="https://www.linkedin.com/in/jyothika-kothapalli-482596323?utm_source=share_via&utm_content=profile&utm_medium=member_android" target="_blank" class="social-link linkedin" aria-label="LinkedIn">
        <svg width="18" height="18" viewBox="0 0 24 24" fill="currentColor"><path d="M16 8a6 6 0 016 6v7h-4v-7a2 2 0 00-2-2 2 2 0 00-2 2v7h-4v-7a6 6 0 016-6zM2 9h4v12H2z"/><circle cx="4" cy="4" r="2"/></svg>
      </a>
      <a href="https://github.com/ammujyothika89" target="_blank" class="social-link github" aria-label="GitHub">
        <svg width="18" height="18" viewBox="0 0 24 24" fill="currentColor"><path d="M12 2C6.477 2 2 6.484 2 12.017c0 4.425 2.865 8.18 6.839 9.504.5.092.682-.217.682-.483 0-.237-.008-.868-.013-1.703-2.782.605-3.369-1.343-3.369-1.343-.454-1.158-1.11-1.466-1.11-1.466-.908-.62.069-.608.069-.608 1.003.07 1.531 1.032 1.531 1.032.892 1.53 2.341 1.088 2.91.832.092-.647.35-1.088.636-1.338-2.22-.253-4.555-1.113-4.555-4.951 0-1.093.39-1.988 1.029-2.688-.103-.253-.446-1.272.098-2.65 0 0 .84-.27 2.75 1.026A9.564 9.564 0 0112 6.844c.85.004 1.705.115 2.504.337 1.909-1.296 2.747-1.027 2.747-1.027.546 1.379.202 2.398.1 2.651.64.7 1.028 1.595 1.028 2.688 0 3.848-2.339 4.695-4.566 4.943.359.309.678.92.678 1.855 0 1.338-.012 2.419-.012 2.747 0 .268.18.58.688.482A10.019 10.019 0 0022 12.017C22 6.484 17.522 2 12 2z"/></svg>
      </a>
    </div>
  </div>
</footer>

<script>
const LOCS = {{ locs | tojson }};
function updateLocs(){
  const city=document.getElementById('city').value;
  const sel=document.getElementById('locality');
  sel.innerHTML='';
  (LOCS[city]||[]).forEach(l=>{const o=document.createElement('option');o.textContent=l;sel.appendChild(o);});
}
updateLocs();

async function predictPrice(){
  const btn=document.querySelector('.predict-btn');
  btn.innerHTML='<i class="ti ti-loader"></i> Predicting...';
  btn.disabled=true;
  const payload={
    city:document.getElementById('city').value,
    locality:document.getElementById('locality').value,
    area:document.getElementById('area').value,
    bhk:document.getElementById('bhk').value,
    bath:document.getElementById('bath').value,
    age:document.getElementById('age').value,
    floor:document.getElementById('floor').value,
    total_floors:document.getElementById('total_floors').value,
    furnishing:document.getElementById('furnish').value,
    parking:document.getElementById('parking').value,
    gym:document.getElementById('gym').value,
    pool:document.getElementById('pool').value
  };
  try{
    const resp=await fetch('/predict',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(payload)});
    const d=await resp.json();
    if(d.error){alert('Error: '+d.error);return;}
    const fmt=v=>v>=10000000?'\u20b9 '+(v/10000000).toFixed(2)+' Cr':'\u20b9 '+(v/100000).toFixed(2)+' L';
    document.getElementById('resultPrice').textContent=fmt(d.price);
    document.getElementById('resultRange').textContent='Confidence range: '+fmt(d.low)+' \u2014 '+fmt(d.high);
    document.getElementById('resSqft').textContent='\u20b9 '+d.ppsf.toLocaleString()+' /sqft';
    document.getElementById('resR2').textContent='R\u00b2 '+d.r2;
    document.getElementById('resMape').textContent='\u00b1'+d.mape+'%';
    document.getElementById('confPct').textContent=Math.round((1-d.mape/100)*100)+'%';
    const box=document.getElementById('resultBox');
    box.classList.remove('show');void box.offsetWidth;box.classList.add('show');
    setTimeout(()=>{document.getElementById('confFill').style.width=Math.round((1-d.mape/100)*100)+'%';},150);
  }catch(e){alert('Cannot reach Flask. Make sure Cell 5 is running.');}
  finally{btn.innerHTML='<i class="ti ti-brain"></i> Predict Price with AI';btn.disabled=false;}
}

function handleSend(btn){
  btn.textContent='\u2713 Message sent!';
  btn.style.background='var(--success)';
  setTimeout(()=>{btn.innerHTML='<i class="ti ti-send"></i> Send Message';btn.style.background='';},3000);
}

const obs=new IntersectionObserver(entries=>entries.forEach(e=>{if(e.isIntersecting)e.target.classList.add('visible');}),{threshold:0.12});
document.querySelectorAll('.fade-up').forEach(el=>obs.observe(el));

const sections=['home','about','accuracy','trust','demo','contact'];
window.addEventListener('scroll',()=>{
  const y=window.scrollY+80;
  sections.forEach(id=>{
    const sec=document.getElementById(id);if(!sec)return;
    const a=document.querySelector('.nav-links a[href="#'+id+'"]');if(!a)return;
    if(y>=sec.offsetTop&&y<sec.offsetTop+sec.offsetHeight){
      document.querySelectorAll('.nav-links a').forEach(x=>x.classList.remove('active'));
      a.classList.add('active');
    }
  });
});

const barSection=document.getElementById('accuracy');
let barsAnimated=false;
const barObs=new IntersectionObserver(entries=>{
  if(entries[0].isIntersecting&&!barsAnimated){
    barsAnimated=true;
    setTimeout(()=>{
      document.getElementById('b1').style.width='{{ (meta.r2*100)|round(1) }}%';
      document.getElementById('b2').style.width='89%';
      document.getElementById('b3').style.width='85%';
      document.getElementById('b4').style.width='72%';
      document.getElementById('b5').style.width='65%';
    },200);
  }
},{threshold:0.3});
barObs.observe(barSection);
</script>
</body>
</html>
"""

# ── CSS (exact from your uploaded file) ──────────────────────────────
CSS = """
*{margin:0;padding:0;box-sizing:border-box}
:root{
  --primary:#1a3c5e;--accent:#2e86de;--accent2:#54a0ff;
  --gold:#f9ca24;--light:#f0f4ff;--white:#ffffff;
  --text:#1a1a2e;--text2:#4a5568;--text3:#718096;
  --border:#e2e8f0;--success:#27ae60;
  --card-bg:#ffffff;--section-alt:#f7faff;
}
html{scroll-behavior:smooth}
body{font-family:'Segoe UI',system-ui,sans-serif;color:var(--text);background:#fff;overflow-x:hidden}
nav{position:fixed;top:0;left:0;right:0;z-index:999;background:rgba(255,255,255,0.96);backdrop-filter:blur(10px);border-bottom:1px solid var(--border);padding:0 2rem;height:64px;display:flex;align-items:center;justify-content:space-between;}
.logo{display:flex;align-items:center;gap:10px;text-decoration:none}
.logo-icon{width:40px;height:40px;border-radius:10px;background:linear-gradient(135deg,var(--primary),var(--accent));display:flex;align-items:center;justify-content:center;color:#fff;font-size:14px;font-weight:700;letter-spacing:-1px;box-shadow:0 2px 8px rgba(46,134,222,0.3);}
.logo-text{font-size:1.25rem;font-weight:700;color:var(--primary);line-height:1}
.logo-text span{font-size:0.7rem;font-weight:400;color:var(--text3);display:block;letter-spacing:0.05em}
.nav-links{display:flex;gap:2rem;list-style:none}
.nav-links a{text-decoration:none;color:var(--text2);font-size:0.9rem;font-weight:500;padding:6px 0;border-bottom:2px solid transparent;transition:all 0.2s}
.nav-links a:hover,.nav-links a.active{color:var(--accent);border-bottom-color:var(--accent)}
.nav-cta{background:var(--accent);color:#fff;padding:8px 20px;border-radius:8px;text-decoration:none;font-size:0.85rem;font-weight:600;transition:background 0.2s,transform 0.1s;border:none;cursor:pointer;display:flex;align-items:center;gap:6px;}
.nav-cta:hover{background:var(--primary);transform:translateY(-1px)}
.hero{min-height:100vh;display:flex;align-items:center;background:linear-gradient(135deg,#0f2c4a 0%,#1a3c5e 40%,#1e4d7b 70%,#2e6da4 100%);position:relative;overflow:hidden;padding:80px 2rem 4rem;}
.hero::before{content:'';position:absolute;inset:0;background:url("data:image/svg+xml,%3Csvg width='60' height='60' viewBox='0 0 60 60' xmlns='http://www.w3.org/2000/svg'%3E%3Cg fill='none' fill-rule='evenodd'%3E%3Cg fill='%23ffffff' fill-opacity='0.03'%3E%3Cpath d='M36 34v-4h-2v4h-4v2h4v4h2v-4h4v-2h-4zm0-30V0h-2v4h-4v2h4v4h2V6h4V4h-4zM6 34v-4H4v4H0v2h4v4h2v-4h4v-2H6zM6 4V0H4v4H0v2h4v4h2V6h4V4H6z'/%3E%3C/g%3E%3C/g%3E%3C/svg%3E");}
.hero-grid{max-width:1100px;margin:0 auto;display:grid;grid-template-columns:1fr 1fr;gap:4rem;align-items:center;width:100%}
.hero-badge{display:inline-flex;align-items:center;gap:6px;background:rgba(255,255,255,0.12);border:1px solid rgba(255,255,255,0.2);color:rgba(255,255,255,0.9);padding:6px 14px;border-radius:20px;font-size:0.78rem;font-weight:500;margin-bottom:1.5rem;letter-spacing:0.04em;}
.hero h1{font-size:3rem;font-weight:700;color:#fff;line-height:1.15;margin-bottom:1.2rem}
.hero h1 .highlight{color:var(--gold)}
.hero-desc{font-size:1.05rem;color:rgba(255,255,255,0.8);line-height:1.75;margin-bottom:2rem;max-width:480px}
.hero-btns{display:flex;gap:1rem;flex-wrap:wrap}
.btn-primary{background:var(--gold);color:var(--primary);padding:12px 28px;border-radius:10px;text-decoration:none;font-weight:700;font-size:0.95rem;transition:all 0.2s;border:none;cursor:pointer;display:inline-flex;align-items:center;gap:6px;}
.btn-primary:hover{background:#f0c400;transform:translateY(-2px)}
.btn-outline{background:transparent;color:#fff;padding:12px 28px;border-radius:10px;text-decoration:none;font-weight:600;font-size:0.95rem;border:2px solid rgba(255,255,255,0.4);transition:all 0.2s;cursor:pointer;display:inline-flex;align-items:center;gap:6px;}
.btn-outline:hover{border-color:#fff;background:rgba(255,255,255,0.1)}
.tech-pills{display:flex;flex-wrap:wrap;gap:8px;margin-top:2rem}
.pill{display:flex;align-items:center;gap:5px;padding:5px 12px;border-radius:20px;font-size:0.75rem;font-weight:600;border:1px solid rgba(255,255,255,0.2);background:rgba(255,255,255,0.08);color:rgba(255,255,255,0.85);}
.hero-visual{position:relative}
.hero-card{background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.15);border-radius:20px;padding:1.5rem;backdrop-filter:blur(10px);}
.house-svg-wrap{width:100%;height:200px;display:flex;align-items:center;justify-content:center;margin-bottom:1.2rem}
.price-result{background:rgba(255,255,255,0.1);border-radius:12px;padding:1rem 1.25rem;margin-bottom:1rem;}
.price-label{font-size:0.75rem;color:rgba(255,255,255,0.6);letter-spacing:0.06em;text-transform:uppercase;margin-bottom:4px}
.price-value{font-size:2rem;font-weight:700;color:var(--gold)}
.price-sub{font-size:0.8rem;color:rgba(255,255,255,0.5);margin-top:2px}
.mini-stats{display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px}
.mini-stat{background:rgba(255,255,255,0.08);border-radius:8px;padding:10px;text-align:center}
.mini-stat-val{font-size:1rem;font-weight:700;color:#fff}
.mini-stat-lbl{font-size:0.68rem;color:rgba(255,255,255,0.5);margin-top:2px}
.float-badge{position:absolute;top:-14px;right:-14px;background:var(--success);color:#fff;padding:6px 12px;border-radius:20px;font-size:0.75rem;font-weight:700;box-shadow:0 4px 12px rgba(39,174,96,0.4);animation:pulse2 2s infinite;display:flex;align-items:center;gap:5px;}
@keyframes pulse2{0%,100%{transform:scale(1)}50%{transform:scale(1.05)}}
section{padding:5rem 2rem}
.container{max-width:1100px;margin:0 auto}
.section-header{text-align:center;margin-bottom:3rem}
.section-tag{display:inline-flex;align-items:center;gap:6px;background:var(--light);color:var(--accent);padding:4px 14px;border-radius:20px;font-size:0.78rem;font-weight:600;letter-spacing:0.05em;margin-bottom:0.75rem;border:1px solid rgba(46,134,222,0.2);}
.section-header h2{font-size:2.2rem;font-weight:700;color:var(--text);margin-bottom:0.75rem}
.section-header p{font-size:1rem;color:var(--text3);max-width:600px;margin:0 auto;line-height:1.7}
.about-section{background:var(--section-alt)}
.about-grid{display:grid;grid-template-columns:1fr 1fr;gap:3rem;align-items:center}
.feature-list{list-style:none;display:flex;flex-direction:column;gap:1rem}
.feature-list li{display:flex;gap:12px;align-items:flex-start;padding:1rem;background:#fff;border-radius:12px;border:1px solid var(--border);transition:transform 0.2s,box-shadow 0.2s;}
.feature-list li:hover{transform:translateY(-2px);box-shadow:0 4px 16px rgba(0,0,0,0.06)}
.feat-icon{width:38px;height:38px;border-radius:10px;background:var(--light);color:var(--accent);display:flex;align-items:center;justify-content:center;font-size:18px;flex-shrink:0;}
.feat-text h4{font-size:0.9rem;font-weight:600;color:var(--text);margin-bottom:3px}
.feat-text p{font-size:0.8rem;color:var(--text3);line-height:1.5}
.accuracy-section{background:#fff}
.accuracy-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:1.5rem;margin-bottom:3rem}
.acc-card{background:var(--card-bg);border:1px solid var(--border);border-radius:16px;padding:1.5rem;text-align:center;transition:transform 0.2s,box-shadow 0.2s;position:relative;overflow:hidden;}
.acc-card::before{content:'';position:absolute;top:0;left:0;right:0;height:4px}
.acc-card.blue::before{background:var(--accent)}.acc-card.green::before{background:var(--success)}
.acc-card.gold::before{background:var(--gold)}.acc-card.purple::before{background:#8e44ad}
.acc-card:hover{transform:translateY(-4px);box-shadow:0 8px 24px rgba(0,0,0,0.08)}
.acc-num{font-size:2.5rem;font-weight:700;line-height:1;margin:0.5rem 0}
.acc-num.blue{color:var(--accent)}.acc-num.green{color:var(--success)}
.acc-num.gold{color:#e67e22}.acc-num.purple{color:#8e44ad}
.acc-label{font-size:0.85rem;font-weight:600;color:var(--text);margin-bottom:4px}
.acc-desc{font-size:0.75rem;color:var(--text3);line-height:1.5}
.acc-icon{font-size:2rem;margin-bottom:0.5rem}
.bar-chart-wrap{background:var(--section-alt);border-radius:16px;padding:2rem;border:1px solid var(--border)}
.bar-chart-title{font-size:1rem;font-weight:600;color:var(--text);margin-bottom:1.5rem;display:flex;align-items:center;gap:8px;}
.bar-row{display:flex;align-items:center;gap:1rem;margin-bottom:1rem}
.bar-name{font-size:0.82rem;color:var(--text2);width:100px;flex-shrink:0;text-align:right}
.bar-track{flex:1;height:28px;background:#e9edf2;border-radius:6px;overflow:hidden;position:relative}
.bar-fill{height:100%;border-radius:6px;display:flex;align-items:center;justify-content:flex-end;padding-right:10px;font-size:0.78rem;font-weight:700;color:#fff;transition:width 1.2s cubic-bezier(0.25,0.8,0.25,1)}
.bar-fill.best{background:linear-gradient(90deg,#2ecc71,#27ae60)}
.bar-fill.good{background:linear-gradient(90deg,var(--accent2),var(--accent))}
.bar-fill.mid{background:linear-gradient(90deg,#fdcb6e,#e17055)}
.bar-fill.low{background:linear-gradient(90deg,#b2bec3,#636e72)}
.trust-section{background:linear-gradient(135deg,#0f2c4a,#1a3c5e);padding:5rem 2rem}
.trust-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:1.5rem}
.trust-card{background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.12);border-radius:16px;padding:1.75rem;text-align:center;transition:background 0.2s,transform 0.2s;}
.trust-card:hover{background:rgba(255,255,255,0.13);transform:translateY(-3px)}
.trust-icon{width:56px;height:56px;border-radius:14px;background:rgba(255,255,255,0.12);color:var(--gold);display:flex;align-items:center;justify-content:center;font-size:26px;margin:0 auto 1rem;}
.trust-card h3{font-size:1rem;font-weight:600;color:#fff;margin-bottom:0.5rem}
.trust-card p{font-size:0.82rem;color:rgba(255,255,255,0.65);line-height:1.6}
.how-section{background:var(--section-alt)}
.steps{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:2rem;position:relative}
.step{text-align:center;padding:1.5rem;position:relative}
.step-num{width:52px;height:52px;border-radius:50%;background:linear-gradient(135deg,var(--accent),var(--primary));color:#fff;font-size:1.2rem;font-weight:700;display:flex;align-items:center;justify-content:center;margin:0 auto 1rem;box-shadow:0 4px 16px rgba(46,134,222,0.35);}
.step h3{font-size:1rem;font-weight:600;color:var(--text);margin-bottom:0.5rem}
.step p{font-size:0.82rem;color:var(--text3);line-height:1.6}
.demo-section{background:#fff}
.predictor{background:var(--section-alt);border:1px solid var(--border);border-radius:20px;padding:2.5rem;max-width:760px;margin:0 auto;}
.form-grid{display:grid;grid-template-columns:1fr 1fr;gap:1.5rem;margin-bottom:1.5rem}
.form-group{display:flex;flex-direction:column;gap:6px}
.form-group label{font-size:0.82rem;font-weight:600;color:var(--text2)}
.form-group input,.form-group select{padding:10px 14px;border:1px solid var(--border);border-radius:8px;font-size:0.9rem;color:var(--text);background:#fff;outline:none;transition:border-color 0.2s;font-family:inherit;}
.form-group input:focus,.form-group select:focus{border-color:var(--accent);box-shadow:0 0 0 3px rgba(46,134,222,0.1)}
.predict-btn{width:100%;padding:14px;background:linear-gradient(135deg,var(--accent),var(--primary));color:#fff;border:none;border-radius:10px;font-size:1rem;font-weight:700;cursor:pointer;transition:opacity 0.2s,transform 0.1s;letter-spacing:0.02em;display:flex;align-items:center;justify-content:center;gap:8px;font-family:inherit;}
.predict-btn:hover{opacity:0.92;transform:translateY(-1px)}
.result-box{margin-top:1.5rem;background:#fff;border:1px solid var(--border);border-radius:12px;padding:1.5rem;display:none;}
.result-box.show{display:block;animation:floatUp 0.4s ease}
.result-price{font-size:2.2rem;font-weight:700;color:var(--accent);margin:0.5rem 0}
.result-range{font-size:0.85rem;color:var(--text3);margin-bottom:1rem}
.confidence-bar-wrap{margin-top:1rem}
.conf-label{font-size:0.8rem;color:var(--text3);margin-bottom:6px;display:flex;justify-content:space-between}
.conf-track{height:10px;background:var(--border);border-radius:5px;overflow:hidden}
.conf-fill{height:100%;border-radius:5px;background:linear-gradient(90deg,var(--accent),var(--success));transition:width 1s ease}
.contact-section{background:var(--section-alt)}
.contact-wrap{display:grid;grid-template-columns:1fr 1fr;gap:3rem;align-items:start;max-width:900px;margin:0 auto}
.contact-info h3{font-size:1.2rem;font-weight:700;color:var(--text);margin-bottom:1rem}
.contact-info p{font-size:0.9rem;color:var(--text3);line-height:1.7;margin-bottom:1.5rem}
.contact-detail{display:flex;align-items:center;gap:10px;margin-bottom:0.75rem;font-size:0.85rem;color:var(--text2)}
.contact-detail i{color:var(--accent);font-size:18px}
.contact-form{background:#fff;border:1px solid var(--border);border-radius:16px;padding:2rem}
.contact-form h3{font-size:1rem;font-weight:700;color:var(--text);margin-bottom:1.25rem;display:flex;align-items:center;gap:8px;}
.cf-group{display:flex;flex-direction:column;gap:6px;margin-bottom:1rem}
.cf-group label{font-size:0.8rem;font-weight:600;color:var(--text2)}
.cf-group input,.cf-group textarea{padding:10px 14px;border:1px solid var(--border);border-radius:8px;font-size:0.88rem;color:var(--text);background:#fff;outline:none;font-family:inherit;transition:border-color 0.2s;}
.cf-group textarea{resize:vertical;min-height:90px}
.cf-group input:focus,.cf-group textarea:focus{border-color:var(--accent);box-shadow:0 0 0 3px rgba(46,134,222,0.08)}
.send-btn{width:100%;padding:12px;background:var(--accent);color:#fff;border:none;border-radius:8px;font-weight:600;font-size:0.9rem;cursor:pointer;transition:background 0.2s,transform 0.1s;font-family:inherit;display:flex;align-items:center;justify-content:center;gap:6px;}
.send-btn:hover{background:var(--primary);transform:translateY(-1px)}
footer{background:var(--text);color:rgba(255,255,255,0.85);padding:3rem 2rem 2rem;}
.footer-grid{max-width:1100px;margin:0 auto;display:grid;grid-template-columns:2fr 1fr 1fr;gap:3rem;margin-bottom:2.5rem}
.footer-brand p{font-size:0.82rem;color:rgba(255,255,255,0.5);line-height:1.7;margin-top:0.75rem;max-width:300px}
.footer-col h4{font-size:0.85rem;font-weight:700;color:#fff;letter-spacing:0.06em;text-transform:uppercase;margin-bottom:1rem}
.footer-col a{display:block;font-size:0.82rem;color:rgba(255,255,255,0.55);text-decoration:none;margin-bottom:0.5rem;transition:color 0.2s}
.footer-col a:hover{color:#fff}
.footer-bottom{max-width:1100px;margin:0 auto;border-top:1px solid rgba(255,255,255,0.1);padding-top:1.5rem;display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:1rem;}
.footer-copy{font-size:0.78rem;color:rgba(255,255,255,0.4)}
.social-links{display:flex;gap:12px}
.social-link{width:40px;height:40px;border-radius:10px;border:1px solid rgba(255,255,255,0.15);display:flex;align-items:center;justify-content:center;color:rgba(255,255,255,0.7);font-size:20px;text-decoration:none;transition:background 0.2s,color 0.2s,border-color 0.2s;}
.social-link:hover{background:rgba(255,255,255,0.12);color:#fff;border-color:rgba(255,255,255,0.35)}
.social-link.github:hover{background:#333;border-color:#333}
.social-link.linkedin:hover{background:#0077b5;border-color:#0077b5}
.scores-row{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:1rem;margin-top:2rem}
.score-tile{padding:1.25rem;border-radius:12px;background:#fff;border:1px solid var(--border);text-align:center;transition:transform 0.2s}
.score-tile:hover{transform:translateY(-3px)}
.score-tile .num{font-size:1.6rem;font-weight:700;color:var(--accent)}
.score-tile .lbl{font-size:0.72rem;color:var(--text3);margin-top:4px}
.score-tile.highlighted{border-color:var(--accent);background:var(--light)}
.score-tile.highlighted .num{color:var(--primary)}
@keyframes floatUp{from{opacity:0;transform:translateY(30px)}to{opacity:1;transform:translateY(0)}}
.fade-up{opacity:0;transform:translateY(40px);transition:opacity 0.6s ease,transform 0.6s ease}
.fade-up.visible{opacity:1;transform:translateY(0)}
@media(max-width:700px){
  .hero-grid,.about-grid,.contact-wrap,.footer-grid{grid-template-columns:1fr}
  .hero-visual{display:none}.hero h1{font-size:2rem}.form-grid{grid-template-columns:1fr}
}
"""

with open('templates/index.html', 'w', encoding='utf-8') as f:
    f.write(HTML.strip())

with open('static/css/style.css', 'w') as f:
    f.write(CSS.strip())

print('✅ templates/index.html written')
print('✅ static/css/style.css written')

✅ templates/index.html written
✅ static/css/style.css written


In [ ]:
import threading, webbrowser, time

# Auto-open browser after 1.5 s
def _open():
    time.sleep(1.5)
    webbrowser.open('http://127.0.0.1:5000')
threading.Thread(target=_open, daemon=True).start()

# Import & run the Flask app
import importlib, sys
if 'app' in sys.modules:
    del sys.modules['app']        # force fresh import each run
import app as flask_app

print(' HomeLens AI is live at  http://127.0.0.1:5000')
print(' Kernel → Interrupt  to stop the server')
flask_app.app.run(debug=False, port=5000, use_reloader=False)

 HomeLens AI is live at  http://127.0.0.1:5000
 Kernel → Interrupt  to stop the server
 * Serving Flask app 'app'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Jun/2026 18:09:53] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:09:54] "GET /static/css/style.css HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:09:54] "GET /static/images/model_analysis.png HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:09:54] "GET /static/images/eda.png HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:09:54] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [13/Jun/2026 18:13:58] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:13:58] "GET /static/css/style.css HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:13:58] "GET /static/images/eda.png HTTP/1.1" 200 -
127.0.0.1 - - [13/Jun/2026 18:13:58] "GET /static/images/model_analysis.png HTTP/1.1" 200 -
